# 01 — Preprocessing

My part of Module 1: turning the raw OSM waterways shapefile into a clean riparian buffer and
river-line layer, for three regions — **Kasarani** (the calibration case study), **Gatharaini**,
and **Motoine**. Covers Steps 1-2 below; Steps 3-4 (Sentinel-2 composite, feature table) are
still open for whoever picks this up next — see the README.

In [1]:
import geopandas as gpd
from shapely.geometry import box

WATERWAYS_PATH = '../data/vectors/gis_osm_waterways_free_1.shp'
OUT_DIR = '../data/processed'

## Step 1 — Define the three regions

One consistent method for all three: a fixed-radius circle around a center point, not a
hand-picked bounding box. A bounding box is tempting but dangerous here — get the extent even
slightly wrong and you're screening a different population of buildings than whatever
ground-truth count you're calibrating against. One radius for every region also keeps them
comparable to each other.

- **Kasarani** — center matches Pamoja Trust's actual field-survey area.
- **Gatharaini / Motoine** — center is the geometric midpoint of the named river itself (no
  independent ground truth exists for either, so there's nothing else to match against).

In [2]:
CASE_STUDY_RADIUS_KM = 3

REGIONS = {
    'Kasarani':   {'method': 'point', 'center': (36.8969, -1.2296)},
    'Gatharaini': {'method': 'river_name', 'river_name': 'Gatharaini River'},
    'Motoine':    {'method': 'river_name', 'river_name': 'Motoine River'},
}


def get_region_center(waterways, region_key):
    cfg = REGIONS[region_key]
    if cfg['method'] == 'point':
        return cfg['center']
    river = waterways[waterways['name'] == cfg['river_name']]
    if river.empty:
        raise ValueError(f"No waterway named {cfg['river_name']!r} found in {WATERWAYS_PATH}")
    midpoint_metric = river.to_crs(epsg=32737).union_all().centroid
    midpoint = gpd.GeoSeries([midpoint_metric], crs=32737).to_crs(epsg=4326).iloc[0]
    return (midpoint.x, midpoint.y)


def get_region_aoi(center, radius_km=CASE_STUDY_RADIUS_KM):
    """A fixed-radius circle, approximated here as its bounding box just for clipping vectors."""
    lon, lat = center
    pad_deg = radius_km / 111.0
    return box(lon - pad_deg, lat - pad_deg, lon + pad_deg, lat + pad_deg)

## Step 2 — Load and clip the waterways to each region

The shapefile is Kenya-wide (47,157 features) — clip to each region's AOI before doing
anything else with it. Real flowing water only: `fclass` in `river`/`stream`, dropping drains
and canals, which aren't what a riparian buffer policy is about.

In [3]:
def load_waterways():
    return gpd.read_file(WATERWAYS_PATH)


def clip_rivers_to_aoi(waterways, aoi):
    clipped = waterways[waterways.intersects(aoi)].copy()
    rivers_only = clipped[clipped['fclass'].isin(['river', 'stream'])].copy()
    if rivers_only.empty:
        raise ValueError('No river/stream features intersect this AOI — check the AOI bounds.')
    return rivers_only